In [3]:
import pandas as pd
import numpy as np
import scipy as sp
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from pybedtools import BedTool
from pathlib import Path
from glob import glob
from multiprocessing import Pool, cpu_count
from typing import List, Tuple, Dict, Mapping, Union, Literal
import pyranges as pr
import numpy.typing as npt
import math
from functools import partial
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

/home/orion/miniconda3/envs/omics/lib/python3.10/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
def read_bed_file(file: str) -> pd.DataFrame:
    return pd.read_csv(file, sep = '\t', 
                       header = None, 
                       names=['Chromosome', 'Start', 'End', 'Score', 'Mindpoint'], 
                       skiprows = 1)

def parse_bed_files(input_files: str, 
                        return_type: Literal["bed", "dataframe"] = "dataframe" 
                       ) -> Union[BedTool, pd.DataFrame]:
    file_paths = glob(input_files, recursive=True)
    with Pool(processes=cpu_count() // 2) as pool:
        dfs = pool.map(read_bed_file, file_paths)
    merged_df = pd.concat(dfs, ignore_index=True).sort_values(by=["Chromosome", "Start"]).reset_index().iloc[:, 1:]
    match return_type:
        case "bed":
            return BedTool.from_dataframe(merged_df)
        case "dataframe":
            return merged_df
        case _:
            raise ValueError(f"Unknown return_type: {return_type}")

rec_rate_10kb = parse_bed_files("../data/processed/recombination_maps/10kb_windows/*", return_type="bed")
rec_rate_10kb.head(10)

In [ ]:
def transform_feature_table(
    gff_object: Union[pd.DataFrame, BedTool, pr.PyRanges],
    chr_pattern: str = r"chr([1-9]|[12][0-9]|3[0-3])$",
    return_type: Literal["bed", "dataframe", "pyranges"] = "dataframe"
) -> Union[BedTool, pd.DataFrame, pr.PyRanges]:
    
    df = _convert_to_dataframe(gff_object)
    _validate_required_columns(df)
    df = _transform_dataframe(df, chr_pattern)
    return _convert_to_output(df, return_type)


def _convert_to_dataframe(gff_object: Union[pd.DataFrame, BedTool, pr.PyRanges]) -> pd.DataFrame:
    if isinstance(gff_object, BedTool):
        return gff_object.to_dataframe()
    elif isinstance(gff_object, pd.DataFrame):
        return gff_object.copy()
    elif isinstance(gff_object, pr.PyRanges):
        return gff_object.df
    else:
        raise TypeError("Input must be a pandas DataFrame, BedTool, or PyRanges object")


def _validate_required_columns(df: pd.DataFrame):
    required = {"chromosome", "start", "end", "# feature", "name", "symbol"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def _transform_dataframe(df: pd.DataFrame, chr_pattern: str) -> pd.DataFrame:
    df = (
        df[["chromosome", "start", "end", "# feature", "name", "symbol"]]
        .assign(
            start=lambda d: d["start"].astype("int64"),
            end=lambda d: d["end"].astype("int64"),
            chromosome=lambda d: "chr" + d["chromosome"].astype(str),
        )
        .loc[lambda d: d["chromosome"].str.match(chr_pattern)]
        .rename(columns={
            "chromosome": "Chromosome",
            "start": "Start",
            "end": "End",
            "# feature": "Feature",
            "name": "Name",
            "symbol": "Symbol"
        })
        .sort_values(by=["Chromosome", "Start"])
        .reset_index(drop=True)
    )
    return df


def _convert_to_output(df: pd.DataFrame, return_type: str) -> Union[BedTool, pd.DataFrame, pr.PyRanges]:
    columns_order = ["Chromosome", "Start", "End", "Feature", "Name", "Symbol"]
    df = df[columns_order]

    match return_type:
        case "bed":
            return BedTool.from_dataframe(df)
        case "dataframe":
            return df
        case "pyranges":
            return pr.PyRanges(df)
        case _:
            raise ValueError(f"Unsupported return_type: {return_type}")

blackcap_feature_table = pd.read_csv("../input/GCF_009819655.1_bSylAtr1.pri_feature_table.txt", sep = '\t', header = 0, dtype=str)

blackcap_gff = transform_feature_table(blackcap_feature_table, return_type="bed")
blackcap_genes = blackcap_gff.filter(lambda x: x.name in ("gene", "mRNA", "CDS")).sort()

In [ ]:
def split_bedtool_by_feature(input_bed: Union[pr.PyRanges, pd.DataFrame, BedTool],
                             return_type: Literal["bed", "dataframe", "pyranges"]) -> dict[str, Union[pr.PyRanges, BedTool, pd.DataFrame]]:
    
    df = _convert_to_dataframe(input_bed)
     
    if df.shape[1] < 4:
        raise ValueError("Expected at least 4 columns in the BedTool Data Format (chrom, start, end, features)")
    return _split_by_feature(df, return_type)


def _split_by_feature(
    df: pd.DataFrame,
    return_type: str
) -> Dict[str, Union[BedTool, pd.DataFrame, pr.PyRanges]]:
    feature_dict: Dict[str, Union[BedTool, pr.PyRanges, pd.DataFrame]] = {}
    feature_col = df.columns[3]  # 4th column: feature

    for feature in df[feature_col].unique():
        df_subset = df[df[feature_col] == feature]

        match return_type:
            case "bed":
                feature_dict[feature] = BedTool.from_dataframe(df_subset)
            case "dataframe":
                feature_dict[feature] = df_subset
            case "pyranges":
                feature_dict[feature] = pr.PyRanges(df_subset)
            case _:
                raise ValueError(f"Unknown return_type: {return_type}")
    
    return feature_dict
blackcap_genes = split_bedtool_by_feature(blackcap_genes, return_type="dataframe")

In [ ]:
blackcap_genome = (
    pd.read_csv("../input/SylAtri_genome.txt.tsv", sep="\t", usecols=['Chromosome name', 'Seq length'])
    .assign(**{'Chromosome name': lambda df: 'chr' + df['Chromosome name'].astype(str)})
    .head(33)
)

In [ ]:
class JaccardCalculator:
    __slots__ = ("chromsizes",)

    def __init__(self, chromsizes: str):
        self.chromsizes = self._load_chromsizes(chromsizes)

    @staticmethod
    def _load_chromsizes(chromsizes: str) -> dict:
        df = pd.read_csv(chromsizes, sep="\t", header=None, names=["chrom", "size"])
        return {row["chrom"]: (0, row["size"]) for _, row in df.iterrows()}

    @staticmethod
    def to_bedtool(obj: Union[pr.PyRanges, BedTool, pd.DataFrame]) -> BedTool:
        if isinstance(obj, BedTool):
            return obj.sort()
        elif isinstance(obj, pr.PyRanges):
            return BedTool.from_dataframe(obj.df).sort()
        elif isinstance(obj, pd.DataFrame):
            return BedTool.from_dataframe(obj).sort()
        else:
            raise TypeError("Input must be PyRanges, BedTool, or pandas DataFrame")

    @staticmethod
    def _single_iter(args):
        bed1_df, bed2_df, chromsizes = args
        bed1 = BedTool.from_dataframe(bed1_df)
        bed2 = BedTool.from_dataframe(bed2_df)
        shuffled_bed1 = bed1.shuffle(genome=chromsizes, chrom=True).sort()
        return shuffled_bed1.jaccard(bed2)["jaccard"]

    def shuffle_intervals(self, bed1_df: pd.DataFrame, bed2_df: pd.DataFrame, iterations: int) -> npt.NDArray[np.float64]:
        args = [(bed1_df, bed2_df, self.chromsizes)] * iterations
        with Pool(processes=cpu_count()) as pool:
            results = pool.map(self._single_iter, args)
        return np.array(results, dtype=np.float64)

    @staticmethod
    def empirical_pvalue_two_tailed(observed: float, null: List[float] | npt.NDArray[np.float64]) -> float:
        if not isinstance(null, np.ndarray):
            null = np.asarray(null, dtype=np.float64)
        null_mean = null.mean()
        diff_obs = abs(observed - null_mean)
        p_val = (np.sum(np.abs(null - null_mean) >= diff_obs) + 1) / (len(null) + 1)
        return p_val

    def compute_jaccard(
        self,
        bed_input1: Union[pr.PyRanges, BedTool, pd.DataFrame],
        bed_input2: Union[pr.PyRanges, BedTool, pd.DataFrame, Mapping[str, Union[pr.PyRanges, BedTool, pd.DataFrame]]],
        iterations: int = 1000
    ) -> pd.DataFrame:

        bed1 = self.to_bedtool(bed_input1)
        bed1_df = bed1.to_dataframe()

        if isinstance(bed_input2, Mapping):
            bed2_dict = {k: self.to_bedtool(v) for k, v in bed_input2.items()}
        else:
            bed2 = self.to_bedtool(bed_input2)
            feature_name = (
                bed_input2.df.iloc[0, 3]
                if isinstance(bed_input2, pr.PyRanges) and bed_input2.df.shape[1] > 3
                else bed_input2.iloc[0, 3]
                if isinstance(bed_input2, pd.DataFrame) and bed_input2.shape[1] > 3
                else "feature"
            )
            bed2_dict = {feature_name: bed2}

        records = []
        for feature, bed2 in bed2_dict.items():
            bed2_df: pd.DataFrame = bed2.to_dataframe()
            observed_jaccard: float = bed1.jaccard(bed2)["jaccard"]
            shuffled_jaccards: npt.NDArray[np.float64] = self.shuffle_intervals(bed1_df, bed2_df, iterations)

            ci_lower: np.float64 = np.percentile(shuffled_jaccards, 2.5)
            ci_upper: np.float64 = np.percentile(shuffled_jaccards, 97.5)
            mean: np.float64 = shuffled_jaccards.mean()
            median: np.float64 = np.median(shuffled_jaccards)
            pval: float = self.empirical_pvalue_two_tailed(observed_jaccard, shuffled_jaccards)

            records.append({
                "feature": feature,
                "observed_jaccard": observed_jaccard,
                "shuffled_jaccard_mean": mean,
                "shuffled_jaccard_median": median,
                "ci_lower_95%": ci_lower,
                "ci_upper_95%": ci_upper,
                "p_value_two_tailed": pval
            })

        return pd.DataFrame(records)

blackcap = JaccardCalculator("../input/blackcap.genome")
jaccard_results = blackcap.compute_jaccard(bed_input1=rec_rate_10kb, bed_input2=blackcap_genes, iterations=1000)

In [ ]:
def pval_to_stars(p: float):
        if p <= 0.0001:
            return "****"
        elif p <= 0.001:
            return "***"
        elif p <= 0.01:
            return "**"
        elif p <= 0.05:
            return "*"
        else:
            return ""
def plot_jaccard_test_results(jaccard_results: pd.DataFrame) -> None:
    jaccard_results['error_lower'] = jaccard_results['shuffled_jaccard_mean'] - jaccard_results['ci_lower_95%']
    jaccard_results['error_upper'] = jaccard_results['ci_upper_95%'] - jaccard_results['shuffled_jaccard_mean']
    jaccard_results["group"] = "Shuffled Recombination Rate Windows"

    fig = px.bar(
        jaccard_results,
        x="feature",
        y="shuffled_jaccard_mean",
        error_y="error_upper",
        error_y_minus="error_lower",
        color="group",
        title="Jaccard Index Comparison of True vs. Shuffled\nRecombination Rate Windows with Genomic Features",
        labels={"feature": "Genomic Feature", "shuffled_jaccard_mean": "Jaccard Index", "group": "Legend"},
        width=900,
        height=600,
    )

    fig.add_trace(go.Bar(
        x = jaccard_results["feature"],
        y = jaccard_results["observed_jaccard"],
        name = "True Recombination Rate Windows",
    ))

    p_values = [
        dict(
            x=row["feature"],
            y=0.55,
            text=pval_to_stars(row['p_value_two_tailed']),
            showarrow=False,
            font=dict(size=20, color="black"),
            yanchor="bottom"
        )
        for _, row in jaccard_results.iterrows()
        if pval_to_stars(row['p_value_two_tailed']) != ""  # Only add annotation if there's a star
    ]

    fig.update_layout(
        barmode="group",
        yaxis=dict(range=[0, 1]),
        xaxis_tickangle=-45,
        annotations=p_values,
        showlegend=True
    )
    fig.show()
    fig.write_html("True_vs_Shuffled_Jaccard.html")
    fig.write_image("True_vs_Shuffled_Jaccard.png", format="png", width = 800, height=600, scale = 3)



In [ ]:
def get_score_strength_per_chrom(input_windows: BedTool | pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not isinstance(input_windows, pd.DataFrame):
        genomic_windows = input_windows.to_dataframe(disable_auto_names=True, 
                                     names=["chrom", "start", "end", "score(cM/Mb)", "midpoint"])
    else:
        genomic_windows = input_windows.iloc[:, :5].copy()
        genomic_windows.columns = ["chrom", "start", "end", "score(cM/Mb)", "midpoint"]
        
    genomic_windows["score_strength"] = pd.qcut(genomic_windows["score(cM/Mb)"], q=3, labels=["low", "medium", "high"])
    min_max_per_label = genomic_windows.groupby("score_strength")["score(cM/Mb)"].agg(['min', 'max']).reset_index()
    min_max_per_label.to_csv("Recombination_Rate_Strength.csv", sep = "\t") 
    counts = genomic_windows.groupby(["chrom", "score_strength"]).size().reset_index(name="count")
    total_per_chrom = genomic_windows.groupby("chrom").size().reset_index(name="total")
    merged = pd.merge(counts, total_per_chrom, on="chrom")
    merged['percent'] = (merged['count'] / merged['total']) * 100
    chrom_order = [f"chr{i}" for i in range(1, 34)]
    merged["chrom"] = pd.Categorical(merged["chrom"], categories=chrom_order, ordered=True)
    return merged.sort_values(['chrom', 'score_strength']), min_max_per_label

score_strength, score_range = get_score_strength_per_chrom(rec_rate_10kb)

In [ ]:
def plot_score_strength_per_chrom(input_data: pd.DataFrame, score_ranges: pd.DataFrame, save_figure: bool = False) -> None:
    from plotly.subplots import make_subplots
    unique_chromosomes = input_data.iloc[:, 0].unique()
    chromosome_number = len(unique_chromosomes)
    pie_number = math.ceil(chromosome_number / 4)
    row_number = pie_number + 1
    column_number = 4
    specs = [[{"type": "pie"}] * column_number for _ in range(pie_number)]
    specs.append([{"type": "table"}, {"type": "table"}, {}, {}])
    subplot_titles = [f"Chromosome {i+1}" for i in range(chromosome_number)] + ["Recombination Rate Strength Range"]
    
    figure = make_subplots(
        rows = row_number, 
        cols = column_number,
        specs=specs,
        subplot_titles=subplot_titles
        )
    
    for i, chrom in enumerate(unique_chromosomes):
        row = (i // column_number) + 1
        col = (i % column_number) + 1
        
        chromosome_data = input_data[input_data.iloc[:, 0] == chrom]
        score_strength = chromosome_data.iloc[:, 1].tolist()
        percent_per_chrom = chromosome_data.iloc[:, 4].tolist()
        
        figure.add_trace(
            go.Pie(
                labels=score_strength,
                values=percent_per_chrom,
                name=f"Chromosome {i+1}",
                automargin=True,
                textposition="inside"
            ),
            row = row,
            col = col
            )
    
    
    figure.add_trace(
        go.Table(
            header = dict(
                values=["Recombination Rate(cM/Mb) Strength", "Lower Bound", "Upper Bound"],
                align ="left",
                fill_color="lightblue",
                font=dict(size=13, color="black")),
            cells=dict(
                values=[score_ranges[col].round(2).tolist() for col in score_ranges.columns],
                align="left",
                font=dict(size=13, color="black")
            ),
        ),
        )
    figure.update_layout(
        height=300 * pie_number,
        width=300 * column_number,
        title_text="Score Strength Percentages per Chromosome",
        showlegend=True  # Turn off if you want individual legends per pie
    )
    if save_figure == True:
        figure.write_html("Rec_Rate_Strength_per_Chromosome.html")
        figure.write_image("Rec_Rate_Strength_per_Chromosome.png", height=1600, width=1200, scale=3)
    
    figure.show()
    
plot_score_strength_per_chrom(score_strength, score_range)

In [ ]:
from pybedtools import BedTool
import pandas as pd
import numpy as np
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr

def compute_gc_content(
  genome_file: str,  
  bed_object: BedTool) -> BedTool:
  
        genome_fna_file = Path(genome_file).expanduser()
        bed = (
            BedTool.from_dataframe(bed_object)
            if isinstance(bed_object, pd.DataFrame)
            else bed_object
        )
        gc_windows = bed.nucleotide_content(fi=genome_fna_file)
        return gc_windows

gc_windows = compute_gc_content(genome_file="../data/interim/blackcap.fasta", bed_object=rec_rate_10kb)
print(gc_windows.head(5))
print(rec_rate_10kb.head(5))


In [ ]:

import dis

def compute_genomic_feature_correlation(bed_feature: BedTool, feature_one_score_idx: int, feature_two_score_idx: int) -> tuple[float, float]:
    
    feature_two_array: np.ndarray = (bed_feature
    .to_dataframe(names=None)
    .iloc[:, feature_two_score_idx]
    .astype(float)
    .to_numpy()
    )
     
    feature_one_array: np.ndarray = (bed_feature
    .to_dataframe(names = None)
    .iloc[:, feature_one_score_idx]
    .astype(float)
    .to_numpy()
    )
        
    return pearsonr(x=feature_one_array, y=feature_two_array, alternative="two-sided")



corr = compute_genomic_feature_correlation(gc_windows, 6, 3)
print(f"pearson correlation coefficient: {corr.correlation:.4f}")
print(f"p value: {corr.pvalue:.4e}")